# R2-03 · Estadística para modelar — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional y de **transferencia**: las mismas herramientas de la lección — correlación, Pearson vs
Spearman, *bootstrap* — sobre otro dato del Estado: la **matrícula escolar 2025** (MINEDUC) cruzada
con la **población** de 30 comunas. ¿Las comunas grandes tienen menos estudiantes por habitante?

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, numpy as np, os, urllib.request
import matplotlib.pyplot as plt
from scipy import stats
for archivo in ["matricula.csv", "comunas.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)
mat = pd.read_csv("matricula.csv")       # matrícula 2025 por establecimiento (MINEDUC)
comunas = pd.read_csv("comunas.csv")     # población de 30 comunas de Chile
print("matrícula:", mat.shape, "| comunas:", comunas.shape)

## 1. Construir el indicador: estudiantes por 1.000 habitantes

Suma la matrícula por comuna, crúzala con la población y calcula la **tasa**. Comparar conteos brutos entre comunas de tamaños tan distintos no dice nada; la tasa sí.

### ✍️ Ejercicio 1 — Calcula la tasa y encuentra los extremos

In [ ]:
est = mat.groupby("comuna")["mat_total"].sum().reset_index()
c = pd.merge(est, comunas, on="comuna")      # el cruce de R2-01
# TODO: tasa = 1000 * mat_total / poblacion
c["tasa"] = ...
# TODO: nombre de la comuna con mayor y con menor tasa (idxmax / idxmin + loc)
comuna_max = ...
comuna_min = ...
print("mayor tasa:", comuna_max, "| menor tasa:", comuna_min)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert len(c) == 30
    assert comuna_max == c.loc[c["tasa"].idxmax(), "comuna"]
    assert comuna_min == c.loc[c["tasa"].idxmin(), "comuna"]
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", '1000 * c["mat_total"] / c["poblacion"]; el nombre con c.loc[c["tasa"].idxmax(), "comuna"].')
    print("   Detalle:", e)

## 2. La correlación que no dice nada: conteos vs tamaño

¿Las comunas con más gente tienen más estudiantes? Obvio. Una correlación altísima entre dos **conteos** suele ser solo el **tamaño** hablando — por eso se trabaja con tasas.

### ✍️ Ejercicio 2 — Correlación entre población y matrícula total

In [ ]:
# TODO: stats.pearsonr(c["poblacion"], c["mat_total"])
r_conteos, p_conteos = ..., ...
print("r(población, matrícula):", r_conteos)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert r_conteos > 0.9, "Dos conteos del mismo territorio correlacionan casi perfecto"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'stats.pearsonr(c["poblacion"], c["mat_total"]) devuelve (r, p).')
    print("   Detalle:", e)

## 3. Pearson vs Spearman: ¿las comunas grandes son menos «escolares»?

Ahora la pregunta interesante: ¿la **tasa** baja cuando la comuna es más grande? Pearson mide relación **lineal** y es sensible a valores extremos; **Spearman** usa rangos (orden) y es robusto. Si discrepan, la relación depende de unos pocos casos.

### ✍️ Ejercicio 3 — Correlación de la tasa con la población, con ambos métodos

In [ ]:
# TODO: Pearson y Spearman entre c["poblacion"] y c["tasa"]
r_p, p_p = ..., ...
rho, p_s = ..., ...
print("Pearson:", r_p, p_p, "| Spearman:", rho, p_s)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert r_p < 0, "Pearson sugiere una relación negativa"
    assert abs(rho) < abs(r_p), "Spearman es más débil: la relación depende de pocos casos"
    assert p_p < 0.05 < p_s
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'stats.pearsonr(...) y stats.spearmanr(...), ambos con (c["poblacion"], c["tasa"]).')
    print("   Detalle:", e)

In [ ]:
# (ilustración) Tasa vs población: ¿quién arrastra la recta?
try:
    plt.figure(figsize=(7, 4))
    plt.scatter(c["poblacion"], c["tasa"], color="#4240e5")
    for _, f in c[(c["poblacion"] > 400000) | (c["tasa"] < 160)].iterrows():
        plt.annotate(f["comuna"], (f["poblacion"], f["tasa"]), fontsize=8,
                     xytext=(4, 4), textcoords="offset points")
    plt.xlabel("Población"); plt.ylabel("Estudiantes por 1.000 habitantes")
    plt.title("30 comunas: tasa de matrícula vs tamaño"); plt.tight_layout(); plt.show()
except (NameError, TypeError, ValueError):
    print("ℹ️ Completa el ejercicio 3 para ver este gráfico.")

## 4. *Bootstrap* de la correlación: ¿cuánto confiar con 30 comunas?

Remuestrea las 30 comunas con reemplazo 2.000 veces y recalcula Pearson en cada una. El ancho del intervalo te dice cuán frágil es la conclusión.

### ✍️ Ejercicio 4 — IC 95% bootstrap del r de Pearson

In [ ]:
rng = np.random.default_rng(42)
rs = []
for _ in range(2000):
    idx = rng.integers(0, len(c), len(c))
    m = c.iloc[idx]
    # TODO: agrega a rs la correlación poblacion–tasa de la remuestra m
    ...
# TODO: percentiles 2,5 y 97,5 de rs -> np.percentile(rs, [2.5, 97.5])
ic_low, ic_high = ..., ...
print("IC95% bootstrap:", ic_low, ic_high)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert len(rs) == 2000
    assert ic_low < r_p < ic_high
    assert ic_high - ic_low > 0.3, "Con 30 comunas el intervalo es ancho"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'rs.append(m["poblacion"].corr(m["tasa"])).')
    print("   Detalle:", e)

## Para pensar: ¿qué mide de verdad la tasa?

La matrícula se cuenta **donde está el colegio**, no donde vive el estudiante. Una comuna con
muchos colegios que reciben alumnos de comunas vecinas tendrá una tasa «inflada», y su vecina
una «desinflada». La correlación con el tamaño puede ser, en parte, un efecto de **cómo se mide**
— otra forma de confusor. Antes de recomendar «más colegios en las comunas grandes», habría que
cruzar con la comuna de residencia de los estudiantes.

---

*Fin de la profundización de R2-03.*